In [13]:
import matplotlib.pyplot as plt
print(plt.style.available)


['Solarize_Light2', '_classic_test_patch', '_mpl-gallery', '_mpl-gallery-nogrid', 'bmh', 'classic', 'dark_background', 'fast', 'fivethirtyeight', 'ggplot', 'grayscale', 'petroff10', 'seaborn-v0_8', 'seaborn-v0_8-bright', 'seaborn-v0_8-colorblind', 'seaborn-v0_8-dark', 'seaborn-v0_8-dark-palette', 'seaborn-v0_8-darkgrid', 'seaborn-v0_8-deep', 'seaborn-v0_8-muted', 'seaborn-v0_8-notebook', 'seaborn-v0_8-paper', 'seaborn-v0_8-pastel', 'seaborn-v0_8-poster', 'seaborn-v0_8-talk', 'seaborn-v0_8-ticks', 'seaborn-v0_8-white', 'seaborn-v0_8-whitegrid', 'tableau-colorblind10']


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
import tensorflow as tf
print(tf.__version__)


2.15.0


In [5]:
plt.style.use('seaborn-v0_8')

In [6]:
import matplotlib.pyplot as plt

plt.style.use('default')
print("OK - no seaborn error")

OK - no seaborn error


In [8]:
import torch
import torchvision
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import tensorflow as tf

%config InlineBackend.figure_format = 'svg'

plt.style.use('seaborn-v0_8')   # FIXED

In [9]:
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)

    except RuntimeError as e:
        print(e)

In [12]:
ratings_data = pd.read_csv(r"C:\Users\Dell\Downloads\ratings.csv")
movie_names_data = pd.read_csv(r"C:\Users\Dell\Downloads\movies.csv")

In [13]:
n_movies = len(movie_names_data)
n_user = len(ratings_data['userId'].unique())

In [14]:
ratings_data = pd.merge(ratings_data, movie_names_data, on='movieId', how='inner')

In [15]:
ratings_data.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [16]:
from sklearn.preprocessing import LabelEncoder
import random
Y = ratings_data.rating
user_enc = LabelEncoder()
movie_enc = LabelEncoder()
X = np.array([user_enc.fit_transform(ratings_data.userId),
              movie_enc.fit_transform(ratings_data.title)]).T

In [17]:
user_enc.classes_[4], movie_enc.classes_[8871]

(5, 'Toy Story (1995)')

In [18]:
for x, y in zip(X[:10], Y[:10]):
    print(list(x), y)

[0, 8871] 4.0
[0, 3661] 4.0
[0, 3845] 4.0
[0, 7523] 5.0
[0, 9119] 5.0
[0, 3252] 3.0
[0, 1284] 5.0
[0, 1337] 4.0
[0, 7180] 5.0
[0, 1535] 5.0


In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)

In [20]:
num_users = len(X)
num_movies = len(X)

In [21]:
from keras.layers import Input, Embedding, Flatten, Dot, Dense, Activation, Dropout
from keras.models import Model

def build_model():
    movie_input = Input(shape=[1], name="Book-Input")
    movie_embedding = Embedding(n_movies+1, 15, name="Book-Embedding")(movie_input)
    movie_vec = Flatten(name="Flatten-Books")(movie_embedding)

    user_input = Input(shape=[1], name="User-Input")
    user_embedding = Embedding(n_user+1, 15, name="User-Embedding")(user_input)
    user_vec = Flatten(name="Flatten-Users")(user_embedding)
    
    prod = Dot(name="Dot-Product", axes=1)([user_vec, movie_vec])
    
    prod = Dense(32)(prod)
    prod = Activation('relu')(prod)
    prod = Dropout(0.5)(prod)

    prod = Dense(16)(prod)
    prod = Activation('relu')(prod)
    prod = Dropout(0.5)(prod)
    prod = Dense(1)(prod)


    model = Model([user_input, movie_input], prod)
    model.compile('adam', 'mean_squared_error', metrics=['accuracy'])

    return model


model = build_model()

In [22]:
model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath='./checkpoint',
    save_weights_only=True,
    monitor='val_loss',
    mode='min',
    save_best_only=True,
    verbose=1)

history = model.fit([X_train[:, 0], X_train[:, 1]], Y_train, 
            epochs=15, 
            verbose=1,
            batch_size=64, 
            validation_data=([X_test[:, 0], X_test[:, 1]], Y_test), 
            callbacks=[model_checkpoint_callback])

Epoch 1/15


1260/1261 [============================>.] - ETA: 0s - loss: 3.3997 - accuracy: 0.0261
Epoch 1: val_loss improved from inf to 1.24736, saving model to .\checkpoint
1261/1261 [==============================] - 23s 15ms/step - loss: 3.3991 - accuracy: 0.0261 - val_loss: 1.2474 - val_accuracy: 0.0280
Epoch 2/15
1261/1261 [==============================] - ETA: 0s - loss: 1.8548 - accuracy: 0.0279
Epoch 2: val_loss improved from 1.24736 to 1.03702, saving model to .\checkpoint
1261/1261 [==============================] - 8s 7ms/step - loss: 1.8548 - accuracy: 0.0279 - val_loss: 1.0370 - val_accuracy: 0.0280
Epoch 3/15
1257/1261 [============================>.] - ETA: 0s - loss: 1.2771 - accuracy: 0.0279
Epoch 3: val_loss improved from 1.03702 to 0.95006, saving model to .\checkpoint
1261/1261 [==============================] - 10s 8ms/step - loss: 1.2767 - accuracy: 0.0279 - val_loss: 0.9501 - val_accuracy: 0.0280
Epoch 4/15
1256/1261 [============================>.] - ETA: 0s

In [23]:
X_test[:5], Y_test[:5]

(array([[ 275, 4337],
        [ 598, 7425],
        [ 482,  334],
        [ 201, 3548],
        [ 273, 3540]], dtype=int64),
 41008    5.0
 94274    2.5
 77380    2.5
 29744    3.0
 40462    4.0
 Name: rating, dtype: float64)

In [24]:
predictions = model.predict([X_test[:5, 0], X_test[:5, 1]])

1/1 [==============================] - 2s 2s/step


In [25]:
print(predictions,"\n\n", Y_test[:5].values)

[[4.130224 ]
 [3.3350282]
 [2.283884 ]
 [4.272288 ]
 [3.4752502]] 

 [5.  2.5 2.5 3.  4. ]


In [26]:
movie_enc.classes_[4]

"'Til There Was You (1997)"

In [28]:
def extract_true_ratings(test_user_id, X_test):
    return X_test[X_test['user_id'] == test_user_id]

In [30]:
print(dir())

['Activation', 'Dense', 'Dot', 'Dropout', 'Embedding', 'Flatten', 'In', 'Input', 'LabelEncoder', 'Model', 'Out', 'X', 'X_test', 'X_train', 'Y', 'Y_test', 'Y_train', '_', '_15', '_17', '_23', '_26', '__', '___', '__builtin__', '__builtins__', '__doc__', '__loader__', '__name__', '__package__', '__spec__', '__vsc_ipynb_file__', '_dh', '_i', '_i1', '_i10', '_i11', '_i12', '_i13', '_i14', '_i15', '_i16', '_i17', '_i18', '_i19', '_i2', '_i20', '_i21', '_i22', '_i23', '_i24', '_i25', '_i26', '_i27', '_i28', '_i29', '_i3', '_i30', '_i4', '_i5', '_i6', '_i7', '_i8', '_i9', '_ih', '_ii', '_iii', '_oh', 'build_model', 'exit', 'extract_true_ratings', 'get_ipython', 'gpus', 'history', 'model', 'model_checkpoint_callback', 'movie_enc', 'movie_names_data', 'n_movies', 'n_user', 'np', 'num_movies', 'num_users', 'open', 'os', 'pd', 'plt', 'predictions', 'quit', 'random', 'ratings_data', 'sns', 'tf', 'torch', 'torchvision', 'train_test_split', 'user_enc', 'x', 'y']


In [2]:
def extract_true_ratings(user_id, X_test):
    return X_test[X_test['user_id'] == user_id]['rating']

In [8]:
def extract_true_ratings(user_id, X_test):
    
    true_ratings = list()
    for x, y in X_test:
        if x == user_id:
            rating = ratings_data[(ratings_data['userId'] == user_enc.classes_[user_id]) \
                & (ratings_data['title'] == movie_enc.classes_[y])]['rating'].values[0]
            true_ratings.append(rating)

    return true_ratings

In [9]:
def predict_ratings(user_id, X_test):
    '''
    given user id predict all ratings for movies
    '''
    user_data = ratings_data[ratings_data['userId'] == user_id]
    movie_ids, movie_names, predictions, movie_genres = list(), list(), list(), list()
    i = 0
    for _id, movie_id in X_test:
        if user_id == X_test[i][0]:
            movie_ids.append(X_test[i, 1])
            movie_names.append(movie_enc.classes_[movie_id])
            pred = model.predict([ np.array([X_test[i, 0]]), np.array([X_test[i, 1]]) ])
            predictions.append(pred[0][0])
        i += 1
    return movie_ids, movie_names, movie_genres, predictions

In [11]:
import pandas as pd

ratings_data = pd.read_csv(r"C:\Users\Dell\Downloads\ratings.csv")
movie_names_data = pd.read_csv(r"C:\Users\Dell\Downloads\movies.csv")

In [12]:
print(ratings_data.head())
print(ratings_data.columns)

   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931
Index(['userId', 'movieId', 'rating', 'timestamp'], dtype='str')


In [13]:
test_user_id = 7
userid_rating_data = ratings_data[ratings_data['userId'] == test_user_id]
userid_rating_data

,userId,movieId,rating,timestamp
874,7,1,4.5,1106635946
875,7,50,4.5,1106635993
876,7,58,3.0,1106635520
877,7,150,4.5,1106635925
878,7,165,4.0,1106635987
...,...,...,...,...
1021,7,48997,2.5,1174263740
1022,7,49272,4.5,1165876367
1023,7,49278,3.5,1174263775
1024,7,49286,0.5,1176181731


In [16]:
from sklearn.model_selection import train_test_split
import pandas as pd

ratings_data = pd.read_csv(r"C:\Users\Dell\Downloads\ratings.csv")

X_train, X_test = train_test_split(ratings_data, test_size=0.2, random_state=42)

In [18]:
import pandas as pd

ratings_data = pd.read_csv(r"C:\Users\Dell\Downloads\ratings.csv")

In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test = train_test_split(ratings_data, test_size=0.2, random_state=42)

In [21]:
test_user_id = 7

In [23]:
for _, row in X_test.iterrows():
    user_id = row['userId']
    movie_id = row['movieId']

In [2]:
def predict_ratings(user_id, X_test):
    movie_ids, movie_names, movie_genres, predictions = [], [], [], []

    for _, row in X_test.iterrows():
        if row['userId'] == user_id:
            movie_ids.append(row['movieId'])
            predictions.append(row['rating'])  # placeholder logic

    movie_names = ["Unknown"] * len(movie_ids)
    movie_genres = ["Unknown"] * len(movie_ids)

    return movie_ids, movie_names, movie_genres, predictions

In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split

ratings_data = pd.read_csv(r"C:\Users\Dell\Downloads\ratings.csv")

In [10]:
X_train, X_test = train_test_split(ratings_data, test_size=0.2, random_state=42)

In [11]:
test_user_id = 7

In [12]:
movie_ids, movie_names, movie_genres, predictions = predict_ratings(test_user_id, X_test)

In [14]:
def extract_true_ratings(test_user_id, X_test):
    return X_test[X_test['userId'] == test_user_id]['rating'].values

In [17]:
dictionary = {
    "user_id": [test_user_id] * len(movie_ids),
    "movie_id": movie_ids,
    "movie_name": movie_names,
    "predicted_ratings": predictions,
    "true_ratings": extract_true_ratings(test_user_id, X_test)
}

In [18]:
prediction_dataframe = pd.DataFrame.from_dict(dictionary, orient='index').transpose()
prediction_dataframe.sort_values('predicted_ratings', ascending=False)

,user_id,movie_id,movie_name,predicted_ratings,true_ratings
1,7,480.0,Unknown,5.0,5.0
8,7,3869.0,Unknown,5.0,5.0
12,7,356.0,Unknown,5.0,5.0
21,7,1240.0,Unknown,5.0,5.0
27,7,5991.0,Unknown,4.5,4.5
11,7,49272.0,Unknown,4.5,4.5
17,7,2671.0,Unknown,4.5,4.5
16,7,8636.0,Unknown,4.5,4.5
30,7,6539.0,Unknown,4.5,4.5
24,7,5816.0,Unknown,4.5,4.5
